In [2]:
# %%
# =============================================================================
# CELL 1: Imports, Setup, and Configuration
# kSZ²–Halo Cross-Correlation Pipeline  (py21cmFAST v4.1.0)
# =============================================================================

import os
import glob
import time
import numpy as np
import matplotlib as mpl
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM

# =============================================================================
# CACHE
# =============================================================================

cache_dir     = "kSZ2_halo_project/cache"
cache_dir_abs = os.path.abspath(cache_dir)
os.makedirs(cache_dir, exist_ok=True)

import py21cmfast as p21c
p21c.config['direc'] = cache_dir_abs

print(f"✓ py21cmfast version : {p21c.__version__}")
print(f"✓ Cache              : {cache_dir_abs}")

# =============================================================================
# OUTPUT DIRECTORIES
# =============================================================================

project_dir = "kSZ2_halo_project"
plot_dir    = f"{project_dir}/plots"
halo_dir    = "lightcone_halos/catalogues"

for d in [project_dir, plot_dir, cache_dir]:
    os.makedirs(d, exist_ok=True)

# =============================================================================
# SIMULATION PARAMETERS  (v4.1.0 verified)
#
# SimulationOptions : box geometry, threads, sampler mass/buffer
# MatterOptions     : KEEP_3D_VELOCITIES, interpolation tables
#                     SAMPLE_METHOD='MASS-LIMITED' by default → halo sampler on
# AstroOptions      : spin temperature, inhomogeneous recombinations
# =============================================================================

z_min         = 5.0
z_max         = 20.0
z_step_factor = 1.02

node_redshifts_custom = np.array(
    p21c.get_logspaced_redshifts(
        min_redshift  = z_min,
        max_redshift  = z_max,
        z_step_factor = z_step_factor,
    )
)

inputs = p21c.InputParameters(
    node_redshifts     = node_redshifts_custom,
    random_seed        = 37,

    simulation_options = p21c.SimulationOptions(
        HII_DIM               = 64,
        BOX_LEN               = 400.0,
        N_THREADS             = 32,
        Z_HEAT_MAX            = 20.0,
        SAMPLER_MIN_MASS      = 1e8,
        SAMPLER_BUFFER_FACTOR = 2.0,
    ),

    matter_options = p21c.MatterOptions(
        KEEP_3D_VELOCITIES       = True,
        USE_INTERPOLATION_TABLES = 'hmf-interpolation',
    ),

    astro_options = p21c.AstroOptions(
        INHOMO_RECO  = True,
        USE_TS_FLUCT = True,
    ),
)

# =============================================================================
# COSMOLOGY
# =============================================================================

cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)

# =============================================================================
# PLOT SETTINGS
# =============================================================================

plt.rcParams.update({
    'font.family'      : 'serif',
    'font.serif'       : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset' : 'cm',
    'font.size'        : 28,
    'axes.labelsize'   : 24,
    'axes.titlesize'   : 24,
    'xtick.labelsize'  : 24,
    'ytick.labelsize'  : 24,
    'legend.fontsize'  : 16,
    'xtick.direction'  : 'in',
    'ytick.direction'  : 'in',
    'xtick.top'        : True,
    'ytick.right'      : True,
    'xtick.major.size' : 6,
    'ytick.major.size' : 6,
    'xtick.minor.size' : 3,
    'ytick.minor.size' : 3,
    'axes.linewidth'   : 1.0,
    'lines.linewidth'  : 1.8,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
})
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

# =============================================================================
# SUMMARY
# =============================================================================

halo_mass_files = sorted(glob.glob(f"{halo_dir}/masses_z*.npy"))

print(f"\n{'='*70}")
print(f"kSZ2-HALO PIPELINE READY  (py21cmfast v{p21c.__version__})")
print(f"{'='*70}")
print(f"  BOX_LEN          : {inputs.simulation_options.BOX_LEN:.1f} Mpc")
print(f"  HII_DIM          : {inputs.simulation_options.HII_DIM}")
print(f"  cell size        : {inputs.simulation_options.cell_size:.3f} Mpc")
print(f"  z range          : {z_min} -> {z_max}  ({len(inputs.node_redshifts)} nodes)")
print(f"  KEEP_3D_VEL      : {inputs.matter_options.KEEP_3D_VELOCITIES}")
print(f"  SAMPLE_METHOD    : {inputs.matter_options.SAMPLE_METHOD}")
#print(f"  has_discrete_halos: {inputs.matter_options.has_discrete_halos}")
print(f"  USE_TS_FLUCT     : {inputs.astro_options.USE_TS_FLUCT}")
print(f"  INHOMO_RECO      : {inputs.astro_options.INHOMO_RECO}")
print(f"  halo catalogues  : {len(halo_mass_files)} snapshots in {halo_dir}/")
print(f"  plots            : {os.path.abspath(plot_dir)}")
print(f"{'='*70}")


✓ py21cmfast version : 4.1.0
✓ Cache              : /user1/swanith/kSZ2_halo_project/cache

kSZ2-HALO PIPELINE READY  (py21cmfast v4.1.0)
  BOX_LEN          : 400.0 Mpc
  HII_DIM          : 64
  cell size        : 6.250 Mpc Mpc
  z range          : 5.0 -> 20.0  (65 nodes)
  KEEP_3D_VEL      : True
  SAMPLE_METHOD    : MASS-LIMITED
  USE_TS_FLUCT     : True
  INHOMO_RECO      : True
  halo catalogues  : 9 snapshots in lightcone_halos/catalogues/
  plots            : /user1/swanith/kSZ2_halo_project/plots


/user1/swanith/miniconda3/envs/p21c_v41/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: Resolution is likely too low for accurate evolved density fields. It is recommended that you either increase the resolution (DIM/BOX_LEN) or set the EVOLVE_DENSITY_LINEARLY flag to True. Got DIM=192, BOX_LEN=400.0, resolution=2.0833333333333335 Mpc Mpc.
  v(inst, attr, value)
/user1/swanith/miniconda3/envs/p21c_v41/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: You are setting R_BUBBLE_MAX != 50 when INHOMO_RECO=True. This is non-standard (but allowed), and usually occurs upon manual update of INHOMO_RECO
  v(inst, attr, value)


In [4]:
# %%
# =============================================================================
# CELL 2 (FINAL): Lightcone Simulation + Field + Halo Array Construction
# Following 21cmFAST's make_lightcone_slices recipe EXACTLY
# =============================================================================

import os
import glob
import time
import h5py
import numpy as np
from astropy.units import pixel

print("\n" + "="*70)
print("CELL 2 — LIGHTCONE + FIELD + HALO ARRAYS (21cmFAST EXACT RECIPE)")
print("="*70)

# =============================================================================
# PATHS & SETUP
# =============================================================================

HALO_OUT        = halo_dir
lightcone_cache = f"{cache_dir}/lightcone.h5"
fields_cache    = f"{cache_dir}/field_arrays.npz"
halos_cache     = f"{cache_dir}/halo_arrays.npz"

BOX_LEN         = float(inputs.simulation_options.BOX_LEN)
HII_DIM         = int(inputs.simulation_options.HII_DIM)
cell_size_mpc   = BOX_LEN / HII_DIM
MASS_CUT        = 10.0**8.5   # M_sun

os.makedirs(HALO_OUT, exist_ok=True)

# =============================================================================
# STEP 1: LIGHTCONE
# =============================================================================

print("\n[1/3] Lightcone")
print("-"*50)
lightconer = p21c.RectilinearLightconer.between_redshifts(
    min_redshift = min(inputs.node_redshifts) + 0.1,
    max_redshift = max(inputs.node_redshifts) - 0.1,
    quantities   = (
        "brightness_temp",
        "density",
        "neutral_fraction",
        "kinetic_temperature",
        "velocity_z",
    ),
    resolution   = inputs.simulation_options.cell_size,
)

if os.path.exists(lightcone_cache):
    print(f"  Loading from cache: {lightcone_cache}")
    lightcone = p21c.LightCone.from_file(lightcone_cache, safe=False)
    print(f"  ✓ Loaded  z=[{lightcone.lightcone_redshifts.min():.2f}, "
          f"{lightcone.lightcone_redshifts.max():.2f}]")
else:
    print(f"  Running simulation...")
    t0 = time.time()
    lightcone = p21c.run_lightcone(
        inputs     = inputs,
        lightconer = lightconer,
        write      = True,
    )
    lightcone.save(lightcone_cache)
    print(f"  ✓ Done in {(time.time()-t0)/60:.1f} min  →  {lightcone_cache}")

z_lc = np.array(lightcone.lightcone_redshifts, dtype=np.float32)
n_lc = len(z_lc)
lc_distances = np.array(lightcone.lightcone_distances.to_value('Mpc'), dtype=np.float32)

print(f"  slices : {n_lc}")
print(f"  fields : {list(lightcone.lightcones.keys())}")
print(f"  lc_distances range: [{lc_distances.min():.1f}, {lc_distances.max():.1f}] Mpc")

# =============================================================================
# STEP 2: EXTRACT FIELD ARRAYS
# =============================================================================

print("\n[2/3] Field arrays")
print("-"*50)

if os.path.exists(fields_cache):
    print(f"  Loading from cache: {fields_cache}")
    d               = np.load(fields_cache)
    density_lc      = d['density_lc']
    neutral_frac_lc = d['neutral_frac_lc']
    los_velocity_lc = d['los_velocity_lc']
    brightness_lc   = d['brightness_lc']
    kinetic_temp_lc = d['kinetic_temp_lc']
    print(f"  ✓ Loaded  shape={density_lc.shape}")
else:
    print(f"  Extracting...")
    t0 = time.time()

    density_lc      = np.array(lightcone.lightcones['density'],
                                dtype=np.float32)
    neutral_frac_lc = np.array(lightcone.lightcones['neutral_fraction'],
                                dtype=np.float32)
    brightness_lc   = np.array(lightcone.lightcones['brightness_temp'],
                                dtype=np.float32)
    los_velocity_lc = np.array(lightcone.lightcones['velocity_z'],
                                dtype=np.float32)
    kinetic_temp_lc = np.array(lightcone.lightcones['kinetic_temperature'],
                                dtype=np.float32)

    np.savez_compressed(
        fields_cache,
        density_lc      = density_lc,
        neutral_frac_lc = neutral_frac_lc,
        los_velocity_lc = los_velocity_lc,
        brightness_lc   = brightness_lc,
        kinetic_temp_lc = kinetic_temp_lc,
        z_lc            = z_lc,
    )
    print(f"  ✓ Done in {time.time()-t0:.1f}s  →  {fields_cache}")

c_Mpc_s     = 299792.458 / 3.08567758e19
vel_rms_kms = los_velocity_lc.std() / c_Mpc_s * 299792.458
print(f"  density_lc      : {density_lc.shape}  mean={density_lc.mean():.3f}")
print(f"  neutral_frac_lc : {neutral_frac_lc.shape}  mean={neutral_frac_lc.mean():.3f}")
print(f"  los_velocity_lc : {los_velocity_lc.shape}  "
      f"std={los_velocity_lc.std():.3e} Mpc/s  (~{vel_rms_kms:.0f} km/s)  ✓")
print(f"  brightness_lc   : {brightness_lc.shape}  mean={brightness_lc.mean():.2f} mK")
print(f"  kinetic_temp_lc : {kinetic_temp_lc.shape}  mean={kinetic_temp_lc.mean():.1f} K")

# =============================================================================
# STEP 3: HALO CATALOGUES - With slab filtering (proper z-layer mapping)
# =============================================================================

print("\n[3/3] Halo catalogues + lightcone arrays")
print("-"*50)

if os.path.exists(halos_cache):
    print(f"  Loading from cache: {halos_cache}")
    d             = np.load(halos_cache)
    halo_mass_lc  = d['halo_mass_lc']
    halo_count_lc = d['halo_count_lc']
    print(f"  ✓ Loaded  shape={halo_mass_lc.shape}  "
          f"filled={np.isfinite(halo_mass_lc).sum():,} pixels")
else:
    print("  Computing initial conditions...")
    t0       = time.time()
    init_box = p21c.compute_initial_conditions(inputs=inputs)
    print(f"  ✓ ICs done in {time.time()-t0:.1f}s")

    node_redshifts_sorted = sorted(inputs.node_redshifts)
    print(f"  Sampling halos at {len(node_redshifts_sorted)} node redshifts...")
    print(f"  Mass cut: 10^{np.log10(MASS_CUT):.1f} M_sun")
    print(f"  Cell size (slab thickness): {cell_size_mpc:.2f} cMpc")

    halo_mass_lc  = np.full((HII_DIM, HII_DIM, n_lc), np.nan, dtype=np.float32)
    halo_count_lc = np.zeros((HII_DIM, HII_DIM, n_lc),         dtype=np.float32)
    
    lc_distances = np.array(lightcone.lightcone_distances.to_value('Mpc'), dtype=np.float32)
    lcpix = lightconer.get_lc_distances_in_pixels(inputs.simulation_options.cell_size)
    t0_total = time.time()

    for i, z_node in enumerate(node_redshifts_sorted):
        halo_cat = p21c.determine_halo_catalog(
            redshift           = z_node,
            initial_conditions = init_box,
            inputs             = inputs,
        )
        pt_halo_cat = p21c.perturb_halo_catalog(
            initial_conditions = init_box,
            inputs             = inputs,
            halo_catalog       = halo_cat,
        )
        
        # Find closest LC index to this redshift
        dc_node = cosmo.comoving_distance(z_node).to_value('Mpc')
        z_idx = np.argmin(np.abs(lc_distances - dc_node))
        
        masses_to_use = pt_halo_cat.get('halo_masses')
        coords_to_use = pt_halo_cat.get('halo_coords')
        
        if masses_to_use is None or len(masses_to_use) == 0:
            print(f"  z={z_node:.3f}  no halos")
            continue
        
        cut = masses_to_use > MASS_CUT
        m_cut = masses_to_use[cut]
        c_cut = coords_to_use[cut]
        
        if len(m_cut) == 0:
            print(f"  z={z_node:.3f}  0 halos above mass cut")
            continue
        
        # ─────────────────────────────────────────────────────────────────────
        # SLAB FILTER: Map LC slice to coeval z-layer (proper mapping)
        # ─────────────────────────────────────────────────────────────────────
        from astropy.units import pixel
        
        lcidx = int((lcpix.max() - lcpix[z_idx] + 1*pixel).to_value(pixel))
        z_cell = (-lcidx + lightconer.index_offset) % HII_DIM
        z_lo = z_cell * cell_size_mpc
        z_hi = z_lo + cell_size_mpc
        
        slab_mask = (c_cut[:, 2] >= z_lo) & (c_cut[:, 2] < z_hi)
        m_slab = m_cut[slab_mask]
        c_slab = c_cut[slab_mask]
        
        if len(m_slab) == 0:
            print(f"  z={z_node:.3f}  0 halos in slab [{z_lo:.2f}, {z_hi:.2f}] cMpc")
            continue
          # ─── ADD HERE: Save raw slab halos for HMF analysis ───
        # ─── Save raw slab halos for HMF analysis ───
        tag = f"z{z_node:.4f}"
        np.save(os.path.join(HALO_OUT, f"masses_{tag}.npy"), m_slab)
        np.save(os.path.join(HALO_OUT, f"coords_{tag}.npy"), c_slab)
        
        # Bin halos at this slice
        mass_map, _, _ = np.histogram2d(
            c_slab[:,0], c_slab[:,1],
            bins=HII_DIM, range=[[0,BOX_LEN],[0,BOX_LEN]],
            weights=m_slab)
        count_map, _, _ = np.histogram2d(
            c_slab[:,0], c_slab[:,1],
            bins=HII_DIM, range=[[0,BOX_LEN],[0,BOX_LEN]])
        
        with np.errstate(invalid='ignore', divide='ignore'):
            avg_map = np.where(count_map > 0,
                              mass_map / count_map, np.nan)
        
        halo_mass_lc[:,:,z_idx]  = avg_map.T.astype(np.float32)
        halo_count_lc[:,:,z_idx] = count_map.T.astype(np.float32)
        
        print(f"  z={z_node:.3f}  {cut.sum():,} total  →  {len(m_slab):,} in slab  "
              f"→  LC idx {z_idx}  z_cell {z_cell}")

    np.savez_compressed(
        halos_cache,
        halo_mass_lc  = halo_mass_lc,
        halo_count_lc = halo_count_lc,
        z_lc          = z_lc,
    )
    print(f"\n  ✓ Built in {(time.time()-t0_total)/60:.1f} min  →  {halos_cache}")

print(f"  halo_mass_lc    : {halo_mass_lc.shape}  "
      f"filled={np.isfinite(halo_mass_lc).sum():,} pixels")
print(f"  halo_count_lc   : {halo_count_lc.shape}  "
      f"max count={halo_count_lc.max():.0f}")


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*70)
print("CELL 2 COMPLETE — all arrays ready")
print("="*70)
print(f"  z_lc            : {n_lc} slices  [{z_lc.min():.2f}, {z_lc.max():.2f}]")
print(f"  density_lc      : {density_lc.shape}")
print(f"  neutral_frac_lc : {neutral_frac_lc.shape}")
print(f"  los_velocity_lc : {los_velocity_lc.shape}  [Mpc/s]  ✓")
print(f"  brightness_lc   : {brightness_lc.shape}")
print(f"  kinetic_temp_lc : {kinetic_temp_lc.shape}")
print(f"  halo_mass_lc    : {halo_mass_lc.shape}  (21cmFAST exact recipe)")
print(f"  halo_count_lc   : {halo_count_lc.shape}  (21cmFAST exact recipe)")
print("="*70)


CELL 2 — LIGHTCONE + FIELD + HALO ARRAYS (21cmFAST EXACT RECIPE)

[1/3] Lightcone
--------------------------------------------------
  Loading from cache: kSZ2_halo_project/cache/lightcone.h5
  ✓ Loaded  z=[5.10, 20.22]
  slices : 477
  fields : ['brightness_temp', 'density', 'kinetic_temperature', 'los_velocity', 'neutral_fraction', 'tau_21', 'velocity_z']
  lc_distances range: [7999.5, 10974.5] Mpc

[2/3] Field arrays
--------------------------------------------------
  Loading from cache: kSZ2_halo_project/cache/field_arrays.npz
  ✓ Loaded  shape=(64, 64, 477)
  density_lc      : (64, 64, 477)  mean=-0.000
  neutral_frac_lc : (64, 64, 477)  mean=0.606
  los_velocity_lc : (64, 64, 477)  std=3.605e-17 Mpc/s  (~1112 km/s)  ✓
  brightness_lc   : (64, 64, 477)  mean=-20.68 mK
  kinetic_temp_lc : (64, 64, 477)  mean=8822.0 K

[3/3] Halo catalogues + lightcone arrays
--------------------------------------------------
  Computing initial conditions...
  ✓ ICs done in 2.6s
  Sampling halos 

In [6]:
import os
import time

# Check halo arrays cache (npz)
halos_cache = "kSZ2_halo_project/cache/halo_arrays.npz"
print(f"halo_arrays.npz: {time.ctime(os.path.getmtime(halos_cache))}")

# Check raw catalogues
halo_dir = "lightcone_halos/catalogues"
files = sorted(os.listdir(halo_dir))
masses_files = [f for f in files if f.startswith('masses')]

if masses_files:
    first_file = os.path.join(halo_dir, masses_files[0])
    last_file = os.path.join(halo_dir, masses_files[-1])
    print(f"First catalogue: {masses_files[0]} - {time.ctime(os.path.getmtime(first_file))}")
    print(f"Last catalogue:  {masses_files[-1]} - {time.ctime(os.path.getmtime(last_file))}")
    print(f"Total catalogues: {len(masses_files)}")

halo_arrays.npz: Wed May 13 22:05:21 2026
First catalogue: masses_z10.0855.npy - Wed May 13 22:04:00 2026
Last catalogue:  masses_z9.8682.npy - Wed May 13 22:03:51 2026
Total catalogues: 65


In [ ]:
# result = p21c.run_lightcone(
#     inputs     = inputs,
#     lightconer = lightconer,
#     write      = True,
# )

# print(f"Type: {type(result)}")
# print(f"Length (if tuple): {len(result) if isinstance(result, tuple) else 'N/A'}")
# if isinstance(result, tuple):
#     for i, item in enumerate(result):
#         print(f"  [{i}] {type(item)}")
# else:
#     print(f"Result: {result}")
#     print(f"Has 'save' method: {hasattr(result, 'save')}")

Type: <class 'tuple'>
Length (if tuple): 4
  [0] <class 'int'>
  [1] <class 'float'>
  [2] <class 'py21cmfast.drivers.coeval.Coeval'>
  [3] <class 'py21cmfast.drivers.lightcone.LightCone'>


In [ ]:
# lc_file = "/user1/swanith/.conda/envs/p21c_v4/lib/python3.11/site-packages/py21cmfast/lightcones.py"

# with open(lc_file) as f:
#     lines = f.readlines()

# print("="*70)
# print("FINDING THE METHOD CONTAINING LINE 341")
# print("="*70)

# # Search backwards from line 341 to find the method definition
# for i in range(340, max(0, 320), -1):
#     if 'def ' in lines[i]:
#         # Found the method, print from here
#         for j in range(i, min(i + 30, len(lines))):
#             print(f"{j:4d}: {lines[j]}", end='')
#         break

FINDING THE METHOD CONTAINING LINE 341
 333:     def coeval_subselect(
 334:         self, lcd: Quantity[pixel], coeval: np.ndarray, coeval_res: Quantity[_LENGTH]
 335:     ):
 336:         """Sub-select the coeval slice corresponding to this coeval distance."""
 337:         # This makes the back of the lightcone exactly line up with the back of the
 338:         # coeval box at that redshift, modulo the index_offset.
 339:         lcpix = self.get_lc_distances_in_pixels(coeval_res)
 340:         lcidx = int((lcpix.max() - lcd + 1 * pixel).to_value(pixel))
 341:         return coeval.take(-lcidx + self.index_offset, axis=2, mode="wrap")
 342: 
 343:     def construct_lightcone(
 344:         self,
 345:         lcd: np.ndarray,
 346:         box: np.ndarray,
 347:     ) -> tuple[np.ndarray, np.ndarray]:
 348:         """Construct slices of the lightcone between two coevals."""
 349:         return box
 350: 
 351:     def construct_los_velocity_lightcone(
 352:         self,
 353:    

In [1]:
import os
cache_dir = "kSZ2_halo_project/cache"
lightcone_cache = f"{cache_dir}/lightcone.h5"

if os.path.exists(lightcone_cache):
    os.remove(lightcone_cache)
    print(f"Deleted: {lightcone_cache}")
else:
    print(f"File not found: {lightcone_cache}")

# Also delete the field arrays cache to be safe
fields_cache = f"{cache_dir}/field_arrays.npz"
if os.path.exists(fields_cache):
    os.remove(fields_cache)
    print(f"Deleted: {fields_cache}")

File not found: kSZ2_halo_project/cache/lightcone.h5


In [ ]:
# import os
# halos_cache = "kSZ2_halo_project/cache/halo_arrays.npz"
# if os.path.exists(halos_cache):
#     os.remove(halos_cache)
#     print(f"Deleted: {halos_cache}")

In [ ]:
# import inspect, py21cmfast as p21c

# # all public methods on the lightconer
# print([m for m in dir(p21c.RectilinearLightconer) if not m.startswith('_')])

# # source of the pixel-distance method — this is the key stitching logic
# print(inspect.getsource(p21c.RectilinearLightconer.get_lc_distances_in_pixels))
# import inspect
# print(inspect.getsource(p21c.RectilinearLightconer.make_lightcone_slices))

# import inspect
# print(inspect.getsource(p21c.RectilinearLightconer.coeval_subselect))

['between_redshifts', 'coeval_subselect', 'construct_lightcone', 'construct_los_velocity_lightcone', 'find_required_lightcone_limits', 'get_lc_distances_in_pixels', 'get_shape', 'lc_redshifts', 'make_lightcone_slices', 'redshift_interpolation', 'validate_options', 'with_equal_cdist_slices']
    def get_lc_distances_in_pixels(self, resolution: Quantity[_LENGTH]):
        """Get the lightcone distances in pixels, given a resolution."""
        return self.lc_distances.to(pixel, pixel_scale(resolution / pixel))

    def make_lightcone_slices(
        self,
        c1: Coeval,  # noqa: F821
        c2: Coeval,  # noqa: F821
    ) -> tuple[dict[str, np.ndarray], np.ndarray]:
        """
        Make lightcone slices out of two coeval objects.

        Parameters
        ----------
        c1, c2 : Coeval
            The coeval boxes to interpolate.

        Returns
        -------
        quantity
            The field names of the quantities required by the lightcone.
        lcidx
       

In [3]:
import os
os.remove("kSZ2_halo_project/cache/halo_arrays.npz")

FileNotFoundError: [Errno 2] No such file or directory: 'kSZ2_halo_project/cache/halo_arrays.npz'

In [22]:
# %%
# =============================================================================
# CELL 3: DIAGNOSTIC PLOTS — using raw halo catalogues
# =============================================================================

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

print("\n" + "="*70)
print("CELL 3 — DIAGNOSTIC PLOTS")
print("="*70)

PLOT_DIR = plot_dir
HALO_OUT = halo_dir
os.makedirs(PLOT_DIR, exist_ok=True)

# Find slices with halos
halo_exists = np.array([np.any(np.isfinite(halo_mass_lc[:,:,i])) for i in range(n_lc)])
idx_with_halos = np.where(halo_exists)[0]

# Pick ~6 slices evenly spaced
if len(idx_with_halos) > 6:
    idx_pick = idx_with_halos[np.linspace(0, len(idx_with_halos)-1, 6, dtype=int)]
else:
    idx_pick = idx_with_halos

z_pick = z_lc[idx_pick]
print(f"\nFound {len(idx_with_halos)} slices with halos, plotting {len(idx_pick)}")

# Get matching node redshifts for each LC index
node_redshifts_sorted = sorted(inputs.node_redshifts)
lc_distances = np.array(lightcone.lightcone_distances.to_value('Mpc'), dtype=np.float32)

def find_node_for_lc_idx(z_idx):
    """Find which node redshift corresponds to this LC index"""
    target_dc = lc_distances[z_idx]
    closest_z_node = min(node_redshifts_sorted, 
                         key=lambda z: abs(cosmo.comoving_distance(z).to_value('Mpc') - target_dc))
    return closest_z_node

# =============================================================================
# PLOT 1: HEAVIEST 2000 HALOS OVER DENSITY (using raw halo positions)
# =============================================================================

print("\n[1/3] Heaviest 2000 halos over density...")
print("-"*50)

with mpl.rc_context(plt.rcParams):
    ncols = 3
    nrows = int(np.ceil(len(idx_pick) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5.5*nrows), constrained_layout=True)
    axes = np.array(axes).flatten()

    for ax, z_idx in zip(axes, idx_pick):
        z_node = z_lc[z_idx]
        
        # Density slice
        dens_slice = density_lc[:, :, z_idx].T
        
        # Density background
        im = ax.imshow(1 + dens_slice, origin='lower', extent=[0, BOX_LEN, 0, BOX_LEN],
                       cmap='Greys', norm=mcolors.LogNorm(vmin=0.1, vmax=10), interpolation='bilinear')
        plt.colorbar(im, ax=ax, label=r'$1+\delta$', fraction=0.046, pad=0.15)
        
        # Load raw halo catalogue for this LC slice
        z_node_match = find_node_for_lc_idx(z_idx)
        tag = f"z{z_node_match:.4f}"
        masses_path = os.path.join(HALO_OUT, f"masses_{tag}.npy")
        coords_path = os.path.join(HALO_OUT, f"coords_{tag}.npy")
        
        n_halos = 0
        if os.path.exists(masses_path):
            m_slab = np.load(masses_path)
            c_slab = np.load(coords_path)
            n_halos = len(m_slab)
            
            if n_halos > 0:
                # Top 2000 heaviest
                n_top = min(2000, n_halos)
                top_idx = np.argsort(m_slab)[-n_top:]
                
                sc = ax.scatter(c_slab[top_idx, 0], c_slab[top_idx, 1],
                                c=np.log10(m_slab[top_idx]),
                                s=10, cmap='plasma', alpha=0.8,
                                vmin=8.5, vmax=10.5, edgecolors='none')
                plt.colorbar(sc, ax=ax, label=r'$\log_{10}(M_\odot)$', fraction=0.046, pad=0.04)
        
        ax.set_xlim(0, BOX_LEN)
        ax.set_ylim(0, BOX_LEN)
        ax.set_xlabel("x  [cMpc]")
        ax.set_ylabel("y  [cMpc]")
        ax.set_title(f"$z = {z_node:.2f}$  ({n_halos:,} halos)", fontsize=12)
    
    for ax in axes[len(idx_pick):]:
        ax.set_visible(False)
    
    fig.suptitle("Heaviest 2000 halos per slice + Density field", fontsize=16, fontweight='bold')
    plt.savefig(os.path.join(PLOT_DIR, "02_heaviest_2000_halos_over_density.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(PLOT_DIR, "02_heaviest_2000_halos_over_density.pdf"), bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 02_heaviest_2000_halos_over_density.png / .pdf")

# =============================================================================
# PLOT 2: ALL HALOS IN SLAB OVER DENSITY
# =============================================================================

print("\n[2/3] All halos in slab over density...")
print("-"*50)

with mpl.rc_context(plt.rcParams):
    ncols = 3
    nrows = int(np.ceil(len(idx_pick) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5.5*nrows), constrained_layout=True)
    axes = np.array(axes).flatten()

    for ax, z_idx in zip(axes, idx_pick):
        z_node = z_lc[z_idx]
        
        # Density slice
        dens_slice = density_lc[:, :, z_idx].T
        
        # Density background
        im = ax.imshow(1 + dens_slice, origin='lower', extent=[0, BOX_LEN, 0, BOX_LEN],
                       cmap='Greys', norm=mcolors.LogNorm(vmin=0.1, vmax=10), interpolation='bilinear')
        plt.colorbar(im, ax=ax, label=r'$1+\delta$', fraction=0.046, pad=0.15)
        
        # Load raw halo catalogue
        z_node_match = find_node_for_lc_idx(z_idx)
        tag = f"z{z_node_match:.4f}"
        masses_path = os.path.join(HALO_OUT, f"masses_{tag}.npy")
        coords_path = os.path.join(HALO_OUT, f"coords_{tag}.npy")
        
        n_halos = 0
        if os.path.exists(masses_path):
            m_slab = np.load(masses_path)
            c_slab = np.load(coords_path)
            n_halos = len(m_slab)
            
            if n_halos > 0:
                sc = ax.scatter(c_slab[:, 0], c_slab[:, 1],
                                c=np.log10(m_slab),
                                s=3, cmap='plasma', alpha=0.6,
                                vmin=8.5, vmax=10.5, edgecolors='none')
                plt.colorbar(sc, ax=ax, label=r'$\log_{10}(M_\odot)$', fraction=0.046, pad=0.04)
        
        ax.set_xlim(0, BOX_LEN)
        ax.set_ylim(0, BOX_LEN)
        ax.set_xlabel("x  [cMpc]")
        ax.set_ylabel("y  [cMpc]")
        ax.set_title(f"$z = {z_node:.2f}$  ({n_halos:,} halos)", fontsize=12)
    
    for ax in axes[len(idx_pick):]:
        ax.set_visible(False)
    
    fig.suptitle("All halos in slab + Density field", fontsize=16, fontweight='bold')
    plt.savefig(os.path.join(PLOT_DIR, "02b_all_halos_in_slab_over_density.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(PLOT_DIR, "02b_all_halos_in_slab_over_density.pdf"), bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 02b_all_halos_in_slab_over_density.png / .pdf")

# =============================================================================
# PLOT 3: LIGHTCONE FIELDS
# =============================================================================

print("\n[3/3] All lightcone fields...")
print("-"*50)

fields_to_plot = [
    ("neutral_frac_lc", r"Neutral Fraction $x_\mathrm{HI}$", "plasma"),
    ("density_lc", r"Matter Density $\delta$", "viridis"),
    ("los_velocity_lc", "LOS Velocity [Mpc/s]", "RdBu_r"),
    ("kinetic_temp_lc", r"Kinetic Temperature $T_k$ [K]", "inferno"),
]

with mpl.rc_context(plt.rcParams):
    n_fields = len(fields_to_plot)
    fig, axes = plt.subplots(n_fields, 1, figsize=(14, 4*n_fields), constrained_layout=True)
    if n_fields == 1:
        axes = [axes]

    for ax, (field_var, label, cmap) in zip(axes, fields_to_plot):
        field_data = eval(field_var)
        mid_x = HII_DIM // 2
        field_slice = field_data[mid_x, :, :]

        im = ax.imshow(field_slice, aspect='auto', origin='lower',
                       cmap=cmap, extent=[z_lc[0], z_lc[-1], 0, BOX_LEN], 
                       interpolation='bilinear')
        plt.colorbar(im, ax=ax, label=label, pad=0.01)
        ax.set_xlabel("Redshift  $z$", fontsize=13)
        ax.set_ylabel("y  [cMpc]", fontsize=13)
        ax.set_title(label, fontsize=12)

    fig.suptitle("Lightcone Fields (y-z slice at mid-x)", fontsize=16, fontweight='bold')
    plt.savefig(os.path.join(PLOT_DIR, "03_all_lightcone_fields.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(PLOT_DIR, "03_all_lightcone_fields.pdf"), bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 03_all_lightcone_fields.png / .pdf")

print("\n" + "="*70)
print("CELL 3 COMPLETE")
print("="*70)


CELL 3 — DIAGNOSTIC PLOTS

Found 65 slices with halos, plotting 6

[1/3] Heaviest 2000 halos over density...
--------------------------------------------------
  ✓ Saved: 02_heaviest_2000_halos_over_density.png / .pdf

[2/3] All halos in slab over density...
--------------------------------------------------
  ✓ Saved: 02b_all_halos_in_slab_over_density.png / .pdf

[3/3] All lightcone fields...
--------------------------------------------------
  ✓ Saved: 03_all_lightcone_fields.png / .pdf

CELL 3 COMPLETE


In [9]:
# =============================================================================
# HMF REDSHIFT EVOLUTION with CAMB + Lightcone Overlay (Fixed Volume)
# =============================================================================
import camb
from astropy.cosmology import FlatLambdaCDM
from scipy.integrate import quad
from scipy.interpolate import interp1d

print("\n[X/4] HMF redshift evolution...")
print("-"*50)

H0     = 67.77
h      = H0 / 100.0
ombh2  = 0.04897 * h**2
omch2  = (0.3086 - 0.04897) * h**2
ns     = 0.9665
sigma8 = 0.8102
Om0    = 0.3086

cosmo_astropy = FlatLambdaCDM(H0=H0, Om0=Om0)
rho_mean_0 = Om0 * cosmo_astropy.critical_density0.to('M_sun/Mpc^3').value

# CAMB setup
pars = camb.CAMBparams()
pars.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, omk=0, tau=0.054)
pars.InitPower.set_params(ns=ns, As=2.1e-9)
pars.set_matter_power(redshifts=[0.0], kmax=1000.0)

results = camb.get_results(pars)
kh, _, pk = results.get_matter_power_spectrum(minkh=1e-4, maxkh=1e4, npoints=500)
pk = pk[0]

def sigma_R_raw(R, kh, pk):
    def integrand(lnk):
        k = np.exp(lnk)
        kR = k * R
        W = 3 * (np.sin(kR) - kR * np.cos(kR)) / kR**3
        pk_interp = np.interp(k / h, kh, pk)
        pk_mpc = pk_interp / h**3
        return k**3 * pk_mpc * W**2 / (2 * np.pi**2)
    val, _ = quad(integrand, np.log(1e-4), np.log(1e4), limit=200)
    return np.sqrt(val)

sigma8_raw = sigma_R_raw(8.0 / h, kh, pk)
norm = sigma8 / sigma8_raw

R_grid = np.logspace(-2, 3, 200)
sigma_grid = np.array([norm * sigma_R_raw(R, kh, pk) for R in R_grid])
sigma_interp = interp1d(np.log(R_grid), np.log(sigma_grid), kind='cubic', fill_value='extrapolate')

def sigma_M(M_msun, z):
    R = (3 * M_msun / (4 * np.pi * rho_mean_0)) ** (1/3)
    s0 = np.exp(sigma_interp(np.log(R)))
    Omz = cosmo_astropy.Om(z)
    gz = (5/2)*Omz / (Omz**(4/7) - (1-Omz) + (1+Omz/2)*(1+(1-Omz)/70))
    g0 = (5/2)*Om0 / (Om0**(4/7) - (1-Om0) + (1+Om0/2)*(1+(1-Om0)/70))
    Dz = gz / (g0 * (1 + z))
    return s0 * Dz

def dlnsigma_dlnM(M_msun, z, dlogM=0.01):
    M1 = M_msun * 10**(dlogM)
    M2 = M_msun * 10**(-dlogM)
    return (np.log(sigma_M(M1,z)) - np.log(sigma_M(M2,z))) / (2*dlogM*np.log(10))

def f_sheth_tormen(nu, a=0.707, p=0.3, A=0.3222):
    nu2 = a * nu**2
    return A * np.sqrt(2*nu2/np.pi) * (1 + nu2**(-p)) * np.exp(-nu2/2)

def hmf_theory(M_msun, z, delta_c=1.686):
    s = sigma_M(M_msun, z)
    nu = delta_c / s
    dlnsdlnM = dlnsigma_dlnM(M_msun, z)
    f = f_sheth_tormen(nu)
    return (rho_mean_0 / M_msun) * f * np.abs(dlnsdlnM)

# Plot HMF evolution with lightcone overlay
M_bins = np.logspace(8.5, 12, 30)
M_cents = 0.5 * (M_bins[:-1] + M_bins[1:])
dlnM = np.diff(np.log(M_bins))

# Sample redshifts
z_sample = np.linspace(5, 20, 8)
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(z_sample)))

with mpl.rc_context(plt.rcParams):
    fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
    
    # Plot theory HMF
    for z, color in zip(z_sample, colors):
        hmf_st = np.array([hmf_theory(M, z) for M in M_cents])
        ax.plot(M_cents, hmf_st, lw=2.5, color=color, label=f'$z={z:.1f}$ (ST)')
    
    # Overlay lightcone HMF from our simulation
    halo_exists = np.array([np.any(np.isfinite(halo_mass_lc[:,:,i])) for i in range(n_lc)])
    idx_with_halos = np.where(halo_exists)[0]
    
    if len(idx_with_halos) > 0:
        # Pick ~6 redshifts with halos
        if len(idx_with_halos) > 6:
            idx_pick = idx_with_halos[np.linspace(0, len(idx_with_halos)-1, 6, dtype=int)]
        else:
            idx_pick = idx_with_halos
        
        z_lc_pick = z_lc[idx_pick]
        colors_lc = plt.cm.Spectral(np.linspace(0.1, 0.9, len(idx_pick)))
        
        for z_idx, color_lc in zip(idx_pick, colors_lc):
            z_node = z_lc[z_idx]
            
            # Get proper slab thickness using neighboring LC slices
            if z_idx > 0 and z_idx < n_lc - 1:
                z_lo = z_lc[z_idx - 1]
                z_hi = z_lc[z_idx + 1]
            else:
                z_lo = z_lc[max(0, z_idx-1)]
                z_hi = z_lc[min(n_lc-1, z_idx+1)]
            
            chi_lo = cosmo_astropy.comoving_distance(z_lo).to('Mpc').value
            chi_hi = cosmo_astropy.comoving_distance(z_hi).to('Mpc').value
            dchi = abs(chi_hi - chi_lo) / 2.0  # Half-interval on each side
            
            # Volume = L^2 × Δχ
            V_slice = BOX_LEN**2 * dchi
            
            # Extract ALL halo masses from this slice (already slab-filtered in Cell 2)
            # Load raw halo catalogue for this redshift
            tag = f"z{z_node:.4f}"
            masses_path = os.path.join(halo_dir, f"masses_{tag}.npy")

            if os.path.exists(masses_path):
                m_all = np.load(masses_path)
            else:
                # Try finding closest matching file
                halo_files = [f for f in os.listdir(halo_dir) if f.startswith('masses_z')]
                available_z = [float(f.replace('masses_z','').replace('.npy','')) for f in halo_files]
                if available_z:
                    closest_z = min(available_z, key=lambda z: abs(z - z_node))
                    m_all = np.load(os.path.join(halo_dir, f"masses_z{closest_z:.4f}.npy"))
                else:
                    continue
            
            if len(m_all) > 0:
                counts, _ = np.histogram(m_all, bins=M_bins)
                hmf_sim = counts / (V_slice * dlnM)
                good = counts >= 5
                
                ax.scatter(M_cents[good], hmf_sim[good], s=50, color=color_lc, 
                          marker='o', alpha=0.6, edgecolors='black', linewidth=0.5,
                          label=f'$z={z_node:.2f}$ (LC sim)')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$M\,[M_\odot]$', fontsize=14)
    ax.set_ylabel(r'$dn/d\ln M$  [cMpc$^{-3}$]', fontsize=14)
    ax.set_title(r"HMF Evolution: Sheth-Tormen (lines) vs Lightcone (points)  |  CAMB P(k)", 
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, ncol=2, framealpha=0.9, loc='best')
    #ax.grid(True, alpha=0.3, which='both')
    
    plt.savefig(os.path.join(plot_dir, "05_hmf_redshift_evolution.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(plot_dir, "05_hmf_redshift_evolution.pdf"), bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 05_hmf_redshift_evolution.png / .pdf")


[X/4] HMF redshift evolution...
--------------------------------------------------


get_matter_power_spectrum using larger k_max than input parameter Transfer.kmax
/var/tmp/pbs.1550061.swarm/ipykernel_988331/1560928843.py:41: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  val, _ = quad(integrand, np.log(1e-4), np.log(1e4), limit=200)
/var/tmp/pbs.1550061.swarm/ipykernel_988331/1560928843.py:41: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  val, _ = quad(integrand, np.log(1e-4), np.log(1e4), limit=200)


  ✓ Saved: 05_hmf_redshift_evolution.png / .pdf


In [ ]:
# %%
# =============================================================================
# CELL 4: Reionization History + Optical Depth Analysis (v4.1.0, single seed)
# Inherited from Cell 1: inputs, cosmo, cache_dir, plot_dir
# Inherited from Cell 2: lightcone, z_lc, neutral_frac_lc
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("CELL 4 — REIONIZATION HISTORY + OPTICAL DEPTH")
print("="*70)

# =============================================================================
# PHYSICAL CONSTANTS  (identical to reference Cell 4)
# =============================================================================

h_little     = 0.6766
Omega_b      = 0.04897468161869667
rho_crit_p_cm3 = 1.88e-29 * h_little**2 / (1.67e-24)
n_H0_cm3     = Omega_b * rho_crit_p_cm3
sigma_T_cm2  = 6.65e-25
cm_per_Mpc   = 3.086e24
n_e0_Mpc3    = n_H0_cm3   * cm_per_Mpc**3
sigma_T_Mpc2 = sigma_T_cm2 / cm_per_Mpc**2
prefactor    = n_e0_Mpc3 * sigma_T_Mpc2

print(f"\nPhysical constants:")
print(f"  n_H0      = {n_H0_cm3:.6e} cm^-3")
print(f"  sigma_T   = {sigma_T_cm2:.6e} cm^2")
print(f"  Prefactor = {prefactor:.6e} Mpc^-1")

# =============================================================================
# GLOBAL IONISATION HISTORY  (v4.1.0 — from lightcone.global_quantities)
# =============================================================================

z_nodes = np.array(lightcone.node_redshifts, dtype=float)

# v4.1.0 key is 'neutral_fraction' (renamed from 'xH_box' in v3)
gq = lightcone.global_quantities
if 'neutral_fraction' in gq:
    xHI_nodes = np.array(gq['neutral_fraction'], dtype=float)
elif 'xH_box' in gq:
    xHI_nodes = np.array(gq['xH_box'], dtype=float)
else:
    raise ValueError(f"No neutral fraction in global_quantities. Keys: {list(gq.keys())}")

# sort low-z → high-z
sort_idx  = np.argsort(z_nodes)
z_nodes   = z_nodes[sort_idx]
xHI_nodes = xHI_nodes[sort_idx]
x_e_nodes = 1.0 - xHI_nodes

z_xe_half = float(np.interp(0.5, x_e_nodes, z_nodes))
print(f"\n  z(x_e = 0.5) = {z_xe_half:.2f}  (midpoint of reionisation)")

# =============================================================================
# OPTICAL DEPTH  (identical formula to reference Cell 4)
# =============================================================================

red_axis = np.array(lightcone.lightcone_redshifts, dtype=float)
pos_axis = np.array(lightcone.lightcone_distances,  dtype=float)

# sort low-z → high-z if needed
if red_axis[0] > red_axis[-1]:
    red_axis = red_axis[::-1]
    pos_axis = pos_axis[::-1]

x_e_interp = np.interp(red_axis, z_nodes, x_e_nodes)

ds_Mpc  = np.abs(np.diff(pos_axis))
z_mid   = 0.5 * (red_axis[:-1] + red_axis[1:])
x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])

dtau      = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds_Mpc
tau       = np.cumsum(dtau)
tau_total = float(tau[-1])

print(f"  tau_total = {tau_total:.6f}")

# store for downstream cells (kSZ integrand needs tau)
tau_results = {
    'red_axis' : red_axis,
    'z_mid'    : z_mid,
    'x_e_mid'  : x_e_mid,
    'ds_Mpc'   : ds_Mpc,
    'dtau'     : dtau,
    'tau'      : tau,
    'tau_total': tau_total,
}

# =============================================================================
# PLOT 3a: Ionisation fraction x_e(z)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

ax.plot(z_nodes, x_e_nodes, color='darkblue', lw=2.5, label=r'$x_e(z)$')
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.7)
ax.axvline(z_xe_half, color='gray', ls='--', lw=1, alpha=0.7)
ax.text(z_xe_half + 0.1, 0.52,
        fr'$z(x_e=0.5)={z_xe_half:.2f}$', fontsize=13)

ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Ionisation Fraction $x_e$')
ax.set_ylim(-0.05, 1.05)
ax.invert_xaxis()
ax.legend(loc='best')

fig.savefig(f"{plot_dir}/xe_history.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir}/xe_history.pdf", bbox_inches='tight')
plt.close(fig)
print(f"\n✓ Saved: xe_history.png / .pdf")

# =============================================================================
# PLOT 3b: Neutral fraction x_HI(z)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

ax.plot(z_nodes, xHI_nodes, color='darkred', lw=2.5, label=r'$x_{\rm HI}(z)$')
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Neutral Fraction $x_{\rm HI}$')
ax.set_ylim(-0.05, 1.05)
ax.invert_xaxis()
ax.legend(loc='best')

fig.savefig(f"{plot_dir}/xHI_history.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir}/xHI_history.pdf", bbox_inches='tight')
plt.close(fig)
print(f"✓ Saved: xHI_history.png / .pdf")

# =============================================================================
# PLOT 3c: Cumulative optical depth tau(z)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

ax.plot(z_mid, tau, color='darkgreen', lw=2.5,
        label=fr'$\tau(<z)$,  $\tau_\mathrm{{total}}={tau_total:.4f}$')
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$')
ax.invert_xaxis()
ax.legend(loc='best')
ax.text(0.05, 0.95,
        f'$\\tau_{{\\rm total}} = {tau_total:.6f}$\n'
        f'seed = {inputs.random_seed}',
        transform=ax.transAxes, fontsize=13,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.savefig(f"{plot_dir}/tau_history.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir}/tau_history.pdf", bbox_inches='tight')
plt.close(fig)
print(f"✓ Saved: tau_history.png / .pdf")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"CELL 3 COMPLETE")
print(f"{'='*70}")
print(f"  z(x_e = 0.5) : {z_xe_half:.2f}")
print(f"  tau_total    : {tau_total:.6f}")
print(f"  tau_results  : ready for kSZ integrand (Cell 4)")
print(f"{'='*70}")


CELL 3 — REIONIZATION HISTORY + OPTICAL DEPTH

Physical constants:
  n_H0      = 2.523928e-07 cm^-3
  sigma_T   = 6.650000e-25 cm^2
  Prefactor = 5.179580e-07 Mpc^-1

  z(x_e = 0.5) = 20.31  (midpoint of reionisation)
  tau_total = 0.038706

✓ Saved: xe_history.png / .pdf
✓ Saved: xHI_history.png / .pdf
✓ Saved: tau_history.png / .pdf

CELL 3 COMPLETE
  z(x_e = 0.5) : 20.31
  tau_total    : 0.038706
  tau_results  : ready for kSZ integrand (Cell 4)


In [11]:
# %%
# =============================================================================
# CELL 5: kSZ Integrand with Visibility Function (v4.1.0, single seed)
#
# kSZ integrand:
#   (1 + delta_b) * x_e * (v_los / c) * exp[-tau(z)]
#
# Inherited from Cell 1 : inputs, cache_dir, plot_dir
# Inherited from Cell 2 : lightcone, z_lc, density_lc, neutral_frac_lc,
#                         los_velocity_lc
# Inherited from Cell 3 : tau_results
# =============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("CELL 5 — kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

# =============================================================================
# CONSTANTS
# =============================================================================

c_Mpc_s = 299792.458 / 3.08567758e19   # speed of light in Mpc/s
z_obs   = 5.0                           # observer redshift

print(f"  c          = {c_Mpc_s:.6e} Mpc/s")
print(f"  z_obs      = {z_obs}")

# =============================================================================
# CACHE
# =============================================================================

integrand_dir   = f"{cache_dir}/kSZ_integrands"
integrand_cache = f"{integrand_dir}/kSZ_integrand_seed{inputs.random_seed}.npy"
os.makedirs(integrand_dir, exist_ok=True)

if os.path.exists(integrand_cache):
    print(f"\n  Loading cached integrand: {integrand_cache}")
    kSZ_integrand = np.load(integrand_cache)
    print(f"  ✓ Loaded  shape={kSZ_integrand.shape}")

else:
    print(f"\n  Computing integrand...")

    # ── redshift axis ─────────────────────────────────────────────────────────
    red_axis_full = np.array(lightcone.lightcone_redshifts, dtype=np.float64)

    # sort low-z → high-z if needed
    if red_axis_full[0] > red_axis_full[-1]:
        red_axis_full = red_axis_full[::-1]

    ind_z = np.where(
        red_axis_full <= float(inputs.simulation_options.Z_HEAT_MAX))[0]

    # ── 3D fields from Cell 2 arrays ─────────────────────────────────────────
    # los_velocity_lc is in Mpc/s (confirmed in Cell 2)
    density_1plus = (1.0 + density_lc     [:, :, ind_z]).astype(np.float64)
    x_e_3D        = (1.0 - neutral_frac_lc[:, :, ind_z]).astype(np.float64)
    v_los_Mpc_s   = los_velocity_lc        [:, :, ind_z] .astype(np.float64)

    print(f"  density_1plus : {density_1plus.shape}  "
          f"mean={density_1plus.mean():.3f}")
    print(f"  x_e_3D        : {x_e_3D.shape}  "
          f"mean={x_e_3D.mean():.3f}")
    print(f"  v_los_Mpc_s   : {v_los_Mpc_s.shape}  "
          f"std={v_los_Mpc_s.std():.3e} Mpc/s")

    # ── interpolate tau(z) onto lightcone redshift grid ──────────────────────
    tr         = tau_results
    red_axis   = np.array(tr['red_axis'], dtype=np.float64)
    tau_array  = np.array(tr['tau'],      dtype=np.float64)
    z_mid_arr  = np.array(tr['z_mid'],    dtype=np.float64)

    tau_extended = np.concatenate([[0.0],        tau_array])
    z_extended   = np.concatenate([[red_axis[0]], z_mid_arr])

    tau_at_lc    = np.interp(red_axis_full[ind_z], z_extended, tau_extended)
    visibility_3D = np.exp(-tau_at_lc)[None, None, :]

    # ── integrand ─────────────────────────────────────────────────────────────
    kSZ_integrand = (density_1plus
                     * x_e_3D
                     * (v_los_Mpc_s / c_Mpc_s)
                     * visibility_3D)

    np.save(integrand_cache, kSZ_integrand)
    print(f"  ✓ Cached → {integrand_cache}")
    print(f"  shape = {kSZ_integrand.shape}")
    print(f"  mean  = {kSZ_integrand.mean():.4e}")
    print(f"  std   = {kSZ_integrand.std():.4e}")
    print(f"  rms   = {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")

# =============================================================================
# PLOT: mid-slice of the integrand cube
# =============================================================================

red_axis_full = np.array(lightcone.lightcone_redshifts, dtype=np.float64)
if red_axis_full[0] > red_axis_full[-1]:
    red_axis_full = red_axis_full[::-1]
ind_z = np.where(
    red_axis_full <= float(inputs.simulation_options.Z_HEAT_MAX))[0]

lc_distances = np.array(lightcone.lightcone_distances, dtype=np.float64)
if lc_distances[0] > lc_distances[-1]:
    lc_distances = lc_distances[::-1]

slice_2D  = kSZ_integrand[:, :, kSZ_integrand.shape[2] // 2]
x_extent  = float(lc_distances[ind_z].max())
y_extent  = float(inputs.simulation_options.BOX_LEN)
vmax      = float(np.percentile(np.abs(kSZ_integrand), 99))

fig, ax = plt.subplots(1, 1, figsize=(12, 5), constrained_layout=True)

im = ax.imshow(
    slice_2D.T,
    extent=[0, x_extent, 0, y_extent],
    aspect='auto',
    cmap='seismic',
    origin='lower',
    vmin=-vmax,
    vmax=vmax,
)
plt.colorbar(im, ax=ax, label=r'kSZ Integrand  $(1+\delta)x_e v_{\rm los}/c\,e^{-\tau}$')
ax.set_xlabel('Comoving Distance [Mpc]')
ax.set_ylabel('Comoving Distance [Mpc]')

# twin redshift axis on top
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
xticks     = ax.get_xticks()
z_at_ticks = np.interp(xticks, lc_distances, red_axis_full)
ax2.set_xticks(xticks)
ax2.set_xticklabels([f'{z:.1f}' for z in z_at_ticks])
ax2.set_xlabel(r'Redshift $z$')

fig.suptitle(
    r'kSZ Integrand: $(1+\delta_b)\,x_e\,v_{\rm los}/c\;e^{-\tau(z)}$',
    fontweight='bold')

fname = f"kSZ_integrand_seed{inputs.random_seed}"
fig.savefig(f"{plot_dir}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"\n✓ Saved: {fname}.png / .pdf")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"CELL 4 COMPLETE")
print(f"{'='*70}")
print(f"  kSZ_integrand : {kSZ_integrand.shape}  [dimensionless per Mpc/s / (Mpc/s)]")
print(f"  rms           : {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")
print(f"  ready for Cell 5 — LoS integration → kSZ map")
print(f"{'='*70}")


CELL 5 — kSZ INTEGRAND WITH VISIBILITY FUNCTION
  c          = 9.715612e-15 Mpc/s
  z_obs      = 5.0

  Loading cached integrand: kSZ2_halo_project/cache/kSZ_integrands/kSZ_integrand_seed37.npy
  ✓ Loaded  shape=(64, 64, 1893)

✓ Saved: kSZ_integrand_seed37.png / .pdf

CELL 4 COMPLETE
  kSZ_integrand : (64, 64, 1893)  [dimensionless per Mpc/s / (Mpc/s)]
  rms           : 1.3689e-03
  ready for Cell 5 — LoS integration → kSZ map


In [12]:
# %%
# =============================================================================
# CELL 6: Line-of-Sight Integrated kSZ Map (v4.1.0, single seed)
#
# kSZ(z_obs) = ∫ dχ [ n_e0 σ_T (1/a²) (1+δ_b) x_e (v_los/c) exp(-τ) ]
#
# Inherited from Cell 1 : inputs, cache_dir, plot_dir
# Inherited from Cell 3 : tau_results
# Inherited from Cell 4 : kSZ_integrand
# =============================================================================

import os
import time
import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("CELL 6 — LINE-OF-SIGHT kSZ MAP INTEGRATION")
print("="*70)

# =============================================================================
# PHYSICAL CONSTANTS (CGS)
# =============================================================================

c_cm_s        = 3.0e10
sigma_T_cm2   = 6.6525e-25
n_e0_cm3      = 2.06e-7
Mpc_to_cm     = 3.0857e24
prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s   # s⁻¹

z_obs = 5.0

print(f"  prefactor = {prefactor_cgs:.4e} s⁻¹")
print(f"  z_obs     = {z_obs:.1f}")

# =============================================================================
# CACHE
# =============================================================================

kSZ_maps_dir = f"{cache_dir}/kSZ_maps"
os.makedirs(kSZ_maps_dir, exist_ok=True)
map_cache = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.1f}_seed{inputs.random_seed}.npy"

if os.path.exists(map_cache):
    print(f"\n  Loading cached kSZ map: {map_cache}")
    kSZ_map = np.load(map_cache)
    print(f"  ✓ Loaded  shape={kSZ_map.shape}  "
          f"RMS={np.sqrt(np.mean(kSZ_map**2)):.4e}")

else:
    if 'kSZ_integrand' not in dir() or kSZ_integrand is None:
        raise RuntimeError("kSZ_integrand not available — run Cell 4 first.")

    print(f"\n  Integrating LoS...")

    # ── axes from tau_results (full lightcone) ────────────────────────────────
    tr       = tau_results
    red_axis = np.array(tr['red_axis'], dtype=np.float64)  # full z axis
    ds_Mpc   = np.array(tr['ds_Mpc'],  dtype=np.float64)  # midpoint spacing
    z_mid    = np.array(tr['z_mid'],   dtype=np.float64)  # midpoint z values
    ds_cm    = ds_Mpc * Mpc_to_cm

    # scale factor at midpoints (same length as z_mid / ds_Mpc)
    a        = 1.0 / (1.0 + red_axis)
    a_sq_mid = 0.5 * (a[:-1]**2 + a[1:]**2)   # shape: (len(red_axis)-1,)

    # ── integration range: z_mid >= z_obs ─────────────────────────────────────
    # Note: kSZ_integrand has shape (HII_DIM, HII_DIM, N_ind_z)
    # where N_ind_z = number of lightcone slices up to Z_HEAT_MAX.
    # z_mid has N_full-1 elements covering the full lightcone.
    # We need to select only the slices that:
    #   (a) are within kSZ_integrand's range  AND
    #   (b) satisfy z_mid >= z_obs
    n_integrand_slices = kSZ_integrand.shape[2]   # N_ind_z - 1 after midpoint

    # midpoint integrand
    kSZ_int_mid = 0.5 * (kSZ_integrand[:, :, :-1]
                          + kSZ_integrand[:, :, 1:])
    # kSZ_int_mid shape: (HII_DIM, HII_DIM, N_ind_z - 1)

    # z_mid is on the full lightcone grid — take the first N slices
    # that correspond to the integrand's coverage (up to Z_HEAT_MAX)
    n_mid      = kSZ_int_mid.shape[2]
    z_mid_int  = z_mid[:n_mid]
    ds_cm_int  = ds_cm[:n_mid]
    a_sq_int   = a_sq_mid[:n_mid]

    # integration range mask
    idx_integrate = np.where(z_mid_int >= z_obs)[0]

    print(f"  z = {z_mid_int[idx_integrate].max():.2f} → "
          f"{z_mid_int[idx_integrate].min():.2f}  "
          f"({len(idx_integrate)} slices)")

    # ── full LoS integrand with prefactor ─────────────────────────────────────
    ds_cm_sel  = ds_cm_int[idx_integrate]
    a_sq_sel   = a_sq_int[idx_integrate]

    kSZ_int_full = ((prefactor_cgs / a_sq_sel[None, None, :])
                    * kSZ_int_mid[:, :, idx_integrate]
                    * (ds_cm_sel / c_cm_s)[None, None, :])

    t0      = time.time()
    kSZ_map = np.sum(kSZ_int_full, axis=2)
    print(f"  ✓ Integration done in {time.time()-t0:.2f}s")

    np.save(map_cache, kSZ_map)
    print(f"  ✓ Cached → {map_cache}")
    print(f"  mean = {kSZ_map.mean():.4e}")
    print(f"  RMS  = {np.sqrt(np.mean(kSZ_map**2)):.4e}")
    print(f"  std  = {kSZ_map.std():.4e}")

# =============================================================================
# PLOT: kSZ map + kSZ² map side by side
# =============================================================================

kSZ2_map      = kSZ_map**2
kSZ2_centered = kSZ2_map - kSZ2_map.mean()

BOX_LEN = float(inputs.simulation_options.BOX_LEN)
extent  = [0, BOX_LEN, 0, BOX_LEN]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

vmax = float(np.percentile(np.abs(kSZ_map), 99))
im0  = axes[0].imshow(kSZ_map.T, origin='lower', extent=extent,
                      cmap='seismic', vmin=-vmax, vmax=vmax)
plt.colorbar(im0, ax=axes[0],
             label=r'$\Delta T_{\rm kSZ}$  [dimensionless]',
             fraction=0.046, pad=0.04)
axes[0].set_xlabel('x  [cMpc]')
axes[0].set_ylabel('y  [cMpc]')
axes[0].set_title(
    r'kSZ map  $\int(1+\delta)\,x_e\,v_{\rm los}/c\;e^{-\tau}\,d\chi$',
    fontweight='bold')

v2  = float(np.percentile(np.abs(kSZ2_centered), 99))
im1 = axes[1].imshow(kSZ2_centered.T, origin='lower', extent=extent,
                     cmap='seismic', vmin=-v2, vmax=v2)
plt.colorbar(im1, ax=axes[1],
             label=r'$(\Delta T_{\rm kSZ})^2 - \langle(\Delta T)^2\rangle$',
             fraction=0.046, pad=0.04)
axes[1].set_xlabel('x  [cMpc]')
axes[1].set_ylabel('y  [cMpc]')
axes[1].set_title(r'kSZ$^2$ map (mean-subtracted)', fontweight='bold')

fname = f"kSZ_map_seed{inputs.random_seed}"
fig.savefig(f"{plot_dir}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"\n✓ Saved: {fname}.png / .pdf")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"CELL 5 COMPLETE")
print(f"{'='*70}")
print(f"  kSZ_map  : {kSZ_map.shape}  "
      f"RMS={np.sqrt(np.mean(kSZ_map**2)):.4e}")
print(f"  kSZ2_map : {kSZ2_map.shape}  "
      f"RMS={np.sqrt(np.mean(kSZ2_map**2)):.4e}")
print(f"  ready for Cell 6 — kSZ² × halo cross-correlation")
print(f"{'='*70}")


CELL 6 — LINE-OF-SIGHT kSZ MAP INTEGRATION
  prefactor = 4.1112e-21 s⁻¹
  z_obs     = 5.0

  Loading cached kSZ map: kSZ2_halo_project/cache/kSZ_maps/kSZ_map_z5.0_seed37.npy
  ✓ Loaded  shape=(64, 64)  RMS=1.1613e-05

✓ Saved: kSZ_map_seed37.png / .pdf

CELL 5 COMPLETE
  kSZ_map  : (64, 64)  RMS=1.1613e-05
  kSZ2_map : (64, 64)  RMS=3.1670e-10
  ready for Cell 6 — kSZ² × halo cross-correlation


In [19]:
# %%
# =============================================================================
# CELL 7: kSZ²–Halo Cross-Correlation Power Spectra (v4.1.0, single seed)
#
# Inherited from Cell 1 : inputs, cache_dir, plot_dir
# Inherited from Cell 2 : lightcone, z_lc, halo_count_lc, halo_mass_lc
# Inherited from Cell 5 : kSZ_map, kSZ2_map
# =============================================================================

import os
import time
import numpy as np

print("\n" + "="*70)
print("CELL 7 — kSZ²–HALO CROSS-CORRELATION POWER SPECTRA")
print("="*70)

# =============================================================================
# MAP PROPERTIES
# =============================================================================

npix_side    = int(inputs.simulation_options.HII_DIM)
box_size_Mpc = float(inputs.simulation_options.BOX_LEN)
pix_size_Mpc = box_size_Mpc / npix_side
pix_area     = pix_size_Mpc**2

print(f"\n  Map      : {npix_side}² pixels")
print(f"  Box      : {box_size_Mpc:.1f} Mpc")
print(f"  Pixel    : {pix_size_Mpc:.3f} Mpc")

# =============================================================================
# k-SPACE GRID
# =============================================================================

dk        = 2 * np.pi / (npix_side * pix_size_Mpc)
kx        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
ky        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
kgrid     = np.sqrt(kx[:, None]**2 + ky[None, :]**2)
k_bins    = np.logspace(np.log10(dk), np.log10(kgrid.max() * 0.9), 35)
k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

print(f"  dk       : {dk:.6f} Mpc⁻¹")
print(f"  k range  : [{kgrid.min():.4f}, {kgrid.max():.4f}] Mpc⁻¹")

# =============================================================================
# CACHE
# =============================================================================

cc_cache = f"{cache_dir}/kSZ2_halo_cross_seed{inputs.random_seed}.npy"

if os.path.exists(cc_cache):
    print(f"\n  Loading cached results: {cc_cache}")
    halo_cross_results = np.load(cc_cache, allow_pickle=True).item()
    print(f"  ✓ Loaded  {len(halo_cross_results)} redshifts")

else:
    if 'kSZ_map' not in dir() or kSZ_map is None:
        raise RuntimeError("kSZ_map not available — run Cell 5 first.")

    # ── kSZ² FFT ──────────────────────────────────────────────────────────────
    kSZ2_map          = kSZ_map**2
    kSZ2_map_centered = kSZ2_map - kSZ2_map.mean()
    fft_kSZ2_shifted  = np.fft.fftshift(np.fft.fft2(kSZ2_map_centered))
    auto_kSZ2_ps2d    = np.abs(fft_kSZ2_shifted)**2 * pix_area / npix_side**2

    print(f"\n  kSZ² RMS : {np.sqrt(np.mean(kSZ2_map**2)):.4e}")
    print(f"  No filter applied (matches reference)")

    # ── Find LC slices that actually have halos ───────────────────────────────
    halo_exists = np.array([halo_count_lc[:,:,i].sum() > 0 for i in range(halo_count_lc.shape[2])])
    lc_indices_with_halos = np.where(halo_exists)[0]
    lc_redshifts = np.array(lightcone.lightcone_redshifts, dtype=np.float64)

    print(f"  Found {len(lc_indices_with_halos)} LC slices with halos")

    halo_cross_results = {}
    loop_start         = time.time()

    for i, idx_closest in enumerate(lc_indices_with_halos):
        z_halo   = float(lc_redshifts[idx_closest])
        z_actual = z_halo
        
        # ── halo count slice (number overdensity) ─────────────────────────────
        halo_slice = halo_count_lc[:, :, idx_closest].astype(np.float64)
        
        halo_mean = halo_slice.mean()
        if halo_mean <= 0:
            continue
        
        delta_h = (halo_slice - halo_mean) / halo_mean
    
    # ... rest of code stays the same
        # ── FFTs ──────────────────────────────────────────────────────────────
        fft_halo_shifted = np.fft.fftshift(np.fft.fft2(delta_h))
        auto_halo_ps2d   = (np.abs(fft_halo_shifted)**2
                            * pix_area / npix_side**2)
        cross_ps2d       = (np.real(np.conj(fft_kSZ2_shifted) * fft_halo_shifted)
                            * pix_area / npix_side**2)

        # ── k-binning ─────────────────────────────────────────────────────────
        C_cross_1d         = np.full(len(k_centers), np.nan)
        C_cross_err_sample = np.full(len(k_centers), np.nan)
        C_cross_err_cosmic = np.full(len(k_centers), np.nan)
        C_cross_err_total  = np.full(len(k_centers), np.nan)
        P_kSZ2_1d          = np.full(len(k_centers), np.nan)
        P_halo_1d          = np.full(len(k_centers), np.nan)
        n_modes            = np.zeros(len(k_centers))

        for j in range(len(k_centers)):
            mask  = (kgrid >= k_bins[j]) & (kgrid < k_bins[j + 1])
            n_pix = int(mask.sum())
            if n_pix > 0:
                cv                   = cross_ps2d[mask]
                C_cross_1d[j]        = np.mean(cv)
                C_cross_err_sample[j]= np.std(cv) / np.sqrt(n_pix)
                P_kSZ2_1d[j]         = np.mean(auto_kSZ2_ps2d[mask])
                P_halo_1d[j]         = np.mean(auto_halo_ps2d[mask])
                k_volume             = (box_size_Mpc / (2 * np.pi))**3
                n_modes[j]           = (4 * np.pi * k_centers[j]**2
                                        * k_volume
                                        * (k_bins[j + 1] - k_bins[j]))
                if n_modes[j] > 0:
                    C_cross_err_cosmic[j] = (
                        np.sqrt(P_kSZ2_1d[j] * P_halo_1d[j]
                                + C_cross_1d[j]**2)
                        / np.sqrt(n_modes[j]))
                C_cross_err_total[j] = np.sqrt(
                    C_cross_err_sample[j]**2
                    + C_cross_err_cosmic[j]**2)

        with np.errstate(divide='ignore', invalid='ignore'):
            r_cross = C_cross_1d / np.sqrt(P_kSZ2_1d * P_halo_1d)

        halo_cross_results[z_halo] = {
            'k_centers'          : k_centers,
            'C_cross_1d'         : C_cross_1d,
            'C_cross_err_sample' : C_cross_err_sample,
            'C_cross_err_cosmic' : C_cross_err_cosmic,
            'C_cross_err_total'  : C_cross_err_total,
            'P_kSZ2_1d'          : P_kSZ2_1d,
            'P_halo_1d'          : P_halo_1d,
            'r_cross'            : r_cross,
            'n_modes'            : n_modes,
            'z_actual'           : z_actual,
            'idx_closest'        : idx_closest,
            'halo_mean'          : halo_mean,
            'halo_rms'           : float(np.sqrt(np.mean(delta_h**2))),
            'kSZ2_rms'           : float(np.sqrt(np.mean(kSZ2_map**2))),
        }

        if (i + 1) % 10 == 0 or i == 0 or i == len(node_redshifts) - 1:
            elapsed = time.time() - loop_start
            eta     = (elapsed / (i + 1)) * (len(node_redshifts) - (i + 1))
            mid     = len(k_centers) // 2
            sign    = '+' if np.isfinite(C_cross_1d[mid]) and C_cross_1d[mid] > 0 else '-'
            print(f"  [{i+1:3d}/{len(node_redshifts)}] z={z_halo:.3f}  "
                  f"sign={sign}  halo_mean={halo_mean:.3e}  ETA={eta:.0f}s")

    np.save(cc_cache, halo_cross_results)
    print(f"\n  ✓ Cached → {cc_cache}")

# =============================================================================
# SUMMARY
# =============================================================================

z_list    = sorted(halo_cross_results.keys())
n_nonzero = sum(1 for z in z_list
                if np.any(np.isfinite(halo_cross_results[z]['C_cross_1d'])))

print(f"\n{'='*70}")
print(f"CELL 6 COMPLETE")
print(f"{'='*70}")
print(f"  Cross-correlations : {len(z_list)} redshifts")
print(f"  Non-empty          : {n_nonzero}")
print(f"  z range            : [{min(z_list):.3f}, {max(z_list):.3f}]")
print(f"  ready for Cell 7 — k→ℓ conversion and D_ℓ plots")
print(f"{'='*70}")


CELL 7 — kSZ²–HALO CROSS-CORRELATION POWER SPECTRA

  Map      : 64² pixels
  Box      : 400.0 Mpc
  Pixel    : 6.250 Mpc
  dk       : 0.015708 Mpc⁻¹
  k range  : [0.0000, 0.7109] Mpc⁻¹

  kSZ² RMS : 3.1670e-10
  No filter applied (matches reference)
  Found 65 LC slices with halos
  [  1/65] z=5.100  sign=+  halo_mean=7.431e+02  ETA=0s
  [ 10/65] z=6.192  sign=+  halo_mean=5.921e+02  ETA=0s
  [ 20/65] z=7.787  sign=-  halo_mean=3.425e+02  ETA=0s
  [ 30/65] z=9.721  sign=-  halo_mean=1.569e+02  ETA=0s
  [ 40/65] z=12.104  sign=+  halo_mean=4.258e+01  ETA=0s
  [ 50/65] z=14.971  sign=+  halo_mean=6.786e+00  ETA=0s
  [ 60/65] z=18.549  sign=-  halo_mean=3.413e-01  ETA=0s
  [ 65/65] z=20.216  sign=-  halo_mean=5.127e-02  ETA=0s

  ✓ Cached → kSZ2_halo_project/cache/kSZ2_halo_cross_seed37.npy

CELL 6 COMPLETE
  Cross-correlations : 65 redshifts
  Non-empty          : 65
  z range            : [5.100, 20.216]
  ready for Cell 7 — k→ℓ conversion and D_ℓ plots


In [18]:
import os
os.remove("kSZ2_halo_project/cache/kSZ2_halo_cross_seed37.npy")

In [21]:
# %%
# =============================================================================
# CELL 8: Visualize kSZ²–Halo Cross-Correlation (v4.1.0, single seed)
#
# Inherited from Cell 1 : inputs, cosmo, plot_dir
# Inherited from Cell 3 : tau_results (for reionisation markers)
# Inherited from Cell 6 : halo_cross_results
# =============================================================================

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

print("\n" + "="*70)
print("CELL 7 — kSZ²–HALO CROSS-CORRELATION VISUALISATION")
print("="*70)

# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================

plot_dir_save = f"{plot_dir}/plot_final_cell"
os.makedirs(plot_dir_save, exist_ok=True)
print(f"  Output : {os.path.abspath(plot_dir_save)}")

# =============================================================================
# LOAD FROM CACHE IF NEEDED
# =============================================================================

if 'halo_cross_results' not in dir() or len(halo_cross_results) == 0:
    cc_cache = f"{cache_dir}/kSZ2_halo_cross_seed{inputs.random_seed}.npy"
    if os.path.exists(cc_cache):
        print(f"  Loading from cache: {cc_cache}")
        halo_cross_results = np.load(cc_cache, allow_pickle=True).item()
        print(f"  ✓ Loaded  {len(halo_cross_results)} redshifts")
    else:
        raise RuntimeError("No cross-correlation results — run Cell 6 first.")
else:
    print(f"  Using in-memory results  ({len(halo_cross_results)} redshifts)")

# =============================================================================
# k → ℓ CONVERSION  (identical scaling to reference Cell 8)
# =============================================================================

print("\n  Converting k → ℓ ...")

h_little = 0.6766

cross_corr_ell = {}

for z_obs in sorted(halo_cross_results.keys()):
    res              = halo_cross_results[z_obs]
    D_A_Mpc          = float(cosmo.angular_diameter_distance(z_obs).value)
    chi_comoving_Mpc = float(cosmo.comoving_distance(z_obs).value)
    k_centers        = res['k_centers']

    ell_from_k       = k_centers * chi_comoving_Mpc / h_little
    scale            = h_little**2 / D_A_Mpc**2

    C_cross_ell      = res['C_cross_1d']            * scale
    C_cross_ell_err  = res['C_cross_err_total']     * scale
    C_sample_ell     = res['C_cross_err_sample']    * scale
    C_cosmic_ell     = res['C_cross_err_cosmic']    * scale

    pref             = ell_from_k * (ell_from_k + 1) / (2 * np.pi)
    D_cross_ell      = pref * C_cross_ell
    D_cross_ell_err  = pref * C_cross_ell_err

    with np.errstate(divide='ignore', invalid='ignore'):
        r_cross = res['r_cross']

    cross_corr_ell[z_obs] = {
        'ell_from_k'     : ell_from_k,
        'D_cross_ell'    : D_cross_ell,
        'D_cross_ell_err': D_cross_ell_err,
        'r_cross'        : r_cross,
    }

print(f"  ✓ Converted {len(cross_corr_ell)} redshifts")

# =============================================================================
# REIONISATION HISTORY  (for vertical markers on r(z) plot)
# =============================================================================

z_nodes  = np.array(sorted(lightcone.node_redshifts), dtype=float)
gq       = lightcone.global_quantities
key      = 'neutral_fraction' if 'neutral_fraction' in gq else 'xH_box'
xHI      = np.array(gq[key], dtype=float)
sort_idx = np.argsort(z_nodes)
z_nodes  = z_nodes[sort_idx]
x_e      = 1.0 - xHI[sort_idx]

def z_at_xe(xe_val):
    return float(np.interp(xe_val, x_e, z_nodes))

# =============================================================================
# SHARED SETTINGS
# =============================================================================

all_z   = np.array(sorted(cross_corr_ell.keys()))
cmap    = mpl.cm.rainbow
norm    = mpl.colors.Normalize(vmin=all_z.min(), vmax=all_z.max())

ell_targets = [500, 1000, 3000]
colors_ell  = ['darkblue', 'darkgreen', 'darkred']

# =============================================================================
# PLOT 1: Rainbow D_ℓ vs ℓ  (one curve per redshift)
# =============================================================================

print("\n=== PLOT 1: Rainbow D_ℓ vs ℓ ===")

fig, ax = plt.subplots(1, 1, figsize=(12, 8), constrained_layout=True)

for z_obs in all_z[::2]:
    res   = cross_corr_ell[z_obs]
    ell   = res['ell_from_k']
    D     = res['D_cross_ell']
    D_err = res['D_cross_ell_err']
    valid = np.isfinite(D) & np.isfinite(D_err) & (ell > 10)
    if valid.sum() > 5:
        color = cmap(norm(z_obs))
        ax.plot(ell[valid], D[valid], color=color, lw=1.5, alpha=0.8)
        ax.fill_between(ell[valid],
                        D[valid] - D_err[valid],
                        D[valid] + D_err[valid],
                        color=color, alpha=0.15)

ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$D_\ell^{\rm kSZ^2 \times h}$  [dimensionless]')
ax.set_title(r'kSZ$^2$–Halo Cross-Power $D_\ell$ vs Redshift',
             fontweight='bold')
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
plt.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$')

fname = "kSZ2_halo_Dl_vs_ell_rainbow"
fig.savefig(f"{plot_dir_save}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"  ✓ Saved: {fname}.png / .pdf")

# =============================================================================
# PLOT 2: D_ℓ vs z at fixed ℓ
# =============================================================================

print("\n=== PLOT 2: D_ℓ vs z at fixed ℓ ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for ell_target, color in zip(ell_targets, colors_ell):
    z_plot, D_plot, D_err_plot = [], [], []
    for z_obs in all_z:
        res = cross_corr_ell[z_obs]
        ell = res['ell_from_k']
        D   = res['D_cross_ell']
        E   = res['D_cross_ell_err']
        idx = int(np.argmin(np.abs(ell - ell_target)))
        if np.isfinite(D[idx]) and np.isfinite(E[idx]):
            z_plot.append(z_obs)
            D_plot.append(D[idx])
            D_err_plot.append(E[idx])
    if len(z_plot) > 2:
        z_plot    = np.array(z_plot)
        D_plot    = np.array(D_plot)
        D_err_plot= np.array(D_err_plot)
        ax.errorbar(z_plot, D_plot, yerr=D_err_plot,
                    color=color, lw=2.5, marker='o', markersize=6,
                    capsize=4, alpha=0.8, label=rf'$\ell={ell_target}$')
        ax.fill_between(z_plot, D_plot - D_err_plot, D_plot + D_err_plot,
                        color=color, alpha=0.2)

ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'$D_\ell^{\rm kSZ^2 \times h}$  [dimensionless]')
ax.set_yscale('symlog', linthresh=1e-12)
ax.set_title(r'kSZ$^2$–Halo Cross-Power Evolution', fontweight='bold')
ax.invert_xaxis()
ax.legend(loc='best')

fname = "kSZ2_halo_Dl_vs_z"
fig.savefig(f"{plot_dir_save}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"  ✓ Saved: {fname}.png / .pdf")

# =============================================================================
# PLOT 3: Correlation coefficient r vs z at fixed ℓ
# =============================================================================

print("\n=== PLOT 3: r vs z at fixed ℓ ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for ell_target, color in zip(ell_targets, colors_ell):
    z_plot, r_plot = [], []
    for z_obs in all_z:
        res = cross_corr_ell[z_obs]
        ell = res['ell_from_k']
        r   = res['r_cross']
        idx = int(np.argmin(np.abs(ell - ell_target)))
        if np.isfinite(r[idx]) and np.abs(r[idx]) < 1.5:
            z_plot.append(z_obs)
            r_plot.append(r[idx])
    if len(z_plot) > 2:
        ax.plot(np.array(z_plot), np.array(r_plot),
                color=color, lw=2.5, marker='o', markersize=5,
                alpha=0.8, label=rf'$\ell={ell_target}$')

# reionisation phase markers
for xe_val, ls in zip([0.2, 0.5, 0.9], [':', '--', ':']):
    z_m = z_at_xe(xe_val)
    ax.axvline(z_m, color='gray', ls=ls, lw=1, alpha=0.6)
    ax.text(z_m + 0.05, 1.05, fr'$x_e={xe_val}$', fontsize=10, color='gray')

ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Correlation Coefficient $r$')
ax.set_title(r'kSZ$^2$–Halo Correlation Coefficient', fontweight='bold')
ax.invert_xaxis()
ax.legend(loc='best')

fname = "kSZ2_halo_r_vs_z"
fig.savefig(f"{plot_dir_save}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"  ✓ Saved: {fname}.png / .pdf")

# =============================================================================
# PLOT 4: Correlation coefficient r vs ℓ (rainbow)
# =============================================================================

print("\n=== PLOT 4: r vs ℓ (rainbow) ===")

fig, ax = plt.subplots(1, 1, figsize=(12, 8), constrained_layout=True)

for z_obs in all_z:
    res   = cross_corr_ell[z_obs]
    ell   = res['ell_from_k']
    r     = res['r_cross']
    valid = np.isfinite(r) & (ell > 10) & (np.abs(r) < 1.5)
    if valid.sum() > 5:
        ax.plot(ell[valid], r[valid],
                color=cmap(norm(z_obs)), lw=1.5, alpha=0.7)

ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
ax.set_xscale('log')
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'Correlation Coefficient $r$')
ax.set_title(r'kSZ$^2$–Halo Correlation Coefficient vs $\ell$',
             fontweight='bold')
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
plt.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$')

fname = "kSZ2_halo_r_vs_ell_rainbow"
fig.savefig(f"{plot_dir_save}/{fname}.png", dpi=300, bbox_inches='tight')
fig.savefig(f"{plot_dir_save}/{fname}.pdf", bbox_inches='tight')
plt.close(fig)
print(f"  ✓ Saved: {fname}.png / .pdf")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"CELL 7 COMPLETE")
print(f"{'='*70}")
print(f"  Plots saved to: {os.path.abspath(plot_dir_save)}")
for fname in ["kSZ2_halo_Dl_vs_ell_rainbow",
              "kSZ2_halo_Dl_vs_z",
              "kSZ2_halo_r_vs_z",
              "kSZ2_halo_r_vs_ell_rainbow"]:
    print(f"    {fname}.png / .pdf")
print(f"{'='*70}")


CELL 7 — kSZ²–HALO CROSS-CORRELATION VISUALISATION
  Output : /user1/swanith/kSZ2_halo_project/plots/plot_final_cell
  Using in-memory results  (65 redshifts)

  Converting k → ℓ ...
  ✓ Converted 65 redshifts

=== PLOT 1: Rainbow D_ℓ vs ℓ ===
  ✓ Saved: kSZ2_halo_Dl_vs_ell_rainbow.png / .pdf

=== PLOT 2: D_ℓ vs z at fixed ℓ ===
  ✓ Saved: kSZ2_halo_Dl_vs_z.png / .pdf

=== PLOT 3: r vs z at fixed ℓ ===
  ✓ Saved: kSZ2_halo_r_vs_z.png / .pdf

=== PLOT 4: r vs ℓ (rainbow) ===
  ✓ Saved: kSZ2_halo_r_vs_ell_rainbow.png / .pdf

CELL 7 COMPLETE
  Plots saved to: /user1/swanith/kSZ2_halo_project/plots/plot_final_cell
    kSZ2_halo_Dl_vs_ell_rainbow.png / .pdf
    kSZ2_halo_Dl_vs_z.png / .pdf
    kSZ2_halo_r_vs_z.png / .pdf
    kSZ2_halo_r_vs_ell_rainbow.png / .pdf
